In [53]:
# Librerías
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bs4 import BeautifulSoup
from astroquery.mpc import MPC

# Cobs API

In [54]:
# Verificar la conexión a internet.
def verificar_conexion():
    try:
        requests.get("http://www.google.com", timeout=5)
        print('✅ Conectado a internet.')
        return True
    
    except requests.ConnectionError:
        print('🛑 Sin conexión a internet.')
        return False

In [55]:
# Conexión con la API de COBS
try:
    content = [] 
    fecha_inicial = '1976-01-01'
    nombre_cometa = '12P'
    Link_cops_API_pagina_1 = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial} 00:00&page=1&exclude_faint=False&exclude_not_accurate=False'

    if True:
        print(f'\n⌛ Conectando con la base de datos [COBS Observaciones].')
        response_pagina_1 = requests.get(Link_cops_API_pagina_1)

        if response_pagina_1.status_code == 200:
            content_pagina_1 = response_pagina_1.json()
            numero_de_paginas = int(content_pagina_1['info']['pages'])

            content.extend(content_pagina_1['objects'])

            for pagina in range(2, numero_de_paginas + 1):
                Link_cops_API_pagina = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial}&page={pagina}&exclude_faint=False&exclude_not_accurate=False'
                response_pagina = requests.get(Link_cops_API_pagina)
                content_pagina = response_pagina.json()
                content.extend(content_pagina['objects'])

            print('✅ Base de datos actualizada [COBS Observaciones].')
        
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response_pagina.status_code}\n{response_pagina.content}')


⌛ Conectando con la base de datos [COBS Observaciones].
✅ Base de datos actualizada [COBS Observaciones].


In [56]:
# Creación del data frame Cometa
cometa_df = pd.DataFrame(content)
cometa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226 entries, 0 to 2225
Data columns (total 47 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   type                       2226 non-null   object 
 1   obs_date                   2226 non-null   object 
 2   comet                      2226 non-null   object 
 3   observer                   2226 non-null   object 
 4   location                   1040 non-null   object 
 5   extinction                 236 non-null    object 
 6   obs_method                 2226 non-null   object 
 7   comet_visibility           6 non-null      object 
 8   magnitude                  2226 non-null   object 
 9   conditions                 88 non-null     object 
 10  ref_catalog                2226 non-null   object 
 11  instrument_aperture        2226 non-null   object 
 12  instrument_type            2226 non-null   object 
 13  instrument_focal_ratio     1496 non-null   float

In [57]:
# Numero de registros y variables sin filtrar la información
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2226
Variables: 47


In [58]:
# Base de datos arrojada por la API
cometa_df.sample(5)

,type,obs_date,comet,observer,location,extinction,obs_method,comet_visibility,magnitude,conditions,...,magnitude_error,comparison_star_magnitude,pixel_size_x,pixel_size_y,pixel_size_unit,obs_comment,obs_sky_quality,obs_sky_quality_method,reference_star_names,date_added
460,V,2024-04-05 02:52:48,"{'type': 'P', 'name': '12P', 'fullname': '12P/...","{'first_name': 'Carl', 'last_name': 'Hergenrot...",None,None,"{'key': 'S', 'name': 'In-Out method', 'source'...",None,4.4,None,...,None,None,None,None,None,,None,NaN,,2024-04-05 15:05:04
651,V,2024-03-23 18:50:00,"{'type': 'P', 'name': '12P', 'fullname': '12P/...","{'first_name': 'Michel', 'last_name': 'Deconin...",Sisteron France,None,"{'key': 'E', 'name': 'Extrafocal-Extinction me...",None,4.8,None,...,None,None,None,None,None,,None,NaN,,2024-03-26 16:03:47
556,C,2024-03-29 10:19:11,"{'type': 'P', 'name': '12P', 'fullname': '12P/...","{'first_name': 'Yoshiaki', 'last_name': 'Yamag...",None,None,"{'key': 'C', 'name': 'Unfiltered total CCD mag...",None,4.8,None,...,None,4.80,3.7,3.7,s,,None,NaN,,2024-04-03 13:10:41
569,V,2024-03-28 19:34:00,"{'type': 'P', 'name': '12P', 'fullname': '12P/...","{'first_name': 'Michel', 'last_name': 'Deconin...","Artignosc-sur-Verdon, Provence, France",None,"{'key': 'E', 'name': 'Extrafocal-Extinction me...",None,4.5,None,...,None,None,None,None,None,4 sketches from 25x to 70x with and without SW...,None,NaN,,2024-03-29 09:55:23
756,C,2024-03-13 20:38:23,"{'type': 'P', 'name': '12P', 'fullname': '12P/...","{'first_name': 'Martin', 'last_name': 'Mašek',...",CTA-N La Palma,None,"{'key': 'k', 'name': 'Kron/Cousins R with CCD'...",None,6.4,None,...,0.03,None,1.6,1.6,s,"Comet Alt. 14.2 Calculated mags 6.5,6.5,6.4,6....",None,NaN,,2025-06-09 10:29:00


In [59]:
# Métodos de observación
cometa_df.obs_method.apply(
    lambda registro: f"{registro['key']}: {registro['name']}" if (registro is not None) and ('key' in registro) and ('name' in registro) else 'Datos faltantes'
).value_counts()

obs_method
S: In-Out method                             643
Z: CCD Visual equivalent                     443
M: Modified-Out method                       414
C: Unfiltered total CCD magnitude            362
V: Johnson/Bessel/Kron/Cousins V with CCD    104
B: Simple Out-Out method                      90
I: In-focus                                   41
D: Johnson/Bessel/Kron/Cousins B with CCD     31
H: Kron/Cousins I with CCD                    26
k: Kron/Cousins R with CCD                    20
c: Unfiltered nuclear CCD magnitude           16
O: Out-of-focus (or extrafocal) method        12
E: Extrafocal-Extinction method               10
-: Unknown                                     7
P: photographic                                4
A: Pogson                                      1
s: VSS method using image intensifier          1
Y: Wratten No. 15 with CCD                     1
Name: count, dtype: int64

In [60]:
# Tratamiento de los datos de interés
cometa_df['obs_method_key'] = cometa_df.obs_method.apply(lambda registro: registro['key'] if registro is not None and 'key' in registro else 'Dato faltante')
cometa_df['obs_date'] = pd.to_datetime(pd.to_datetime(cometa_df.obs_date).dt.date) # type: ignore
cometa_df['magnitude'] = pd.to_numeric(cometa_df.magnitude)

In [61]:
# Creación del data frame curva de luz cruda
curva_de_luz_cruda_df = cometa_df[['obs_method_key', 'obs_date', 'magnitude']].copy()
curva_de_luz_cruda_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226 entries, 0 to 2225
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   obs_method_key  2226 non-null   object        
 1   obs_date        2226 non-null   datetime64[ns]
 2   magnitude       2226 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 52.3+ KB


In [62]:
# Numero de registros y variables con la información filtrada
filas,columnas = curva_de_luz_cruda_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2226
Variables: 3


In [63]:
# Data Frame de la curva de luz
curva_de_luz_cruda_df.sample(5)

,obs_method_key,obs_date,magnitude
827,M,2024-03-08,5.9
1266,S,2024-01-09,9.8
1834,V,2023-09-11,11.9
1009,S,2024-02-19,7.4
302,M,2024-04-26,4.7


In [64]:
# Curva de luz cruda.
labels = {'obs_date':'Observation Date','magnitude':'Apparent total magnitude', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_cruda_df, x='obs_date', y='magnitude', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Perihelio Cobs API

In [65]:
# Conexión con la API de COBS para obtener el perihelio
try: 
    Link_cops_API = f'https://cobs.si/api/comet.api?des={nombre_cometa}'

    if verificar_conexion():
        response = requests.get(Link_cops_API)

        if response.status_code == 200:
            perihelio = pd.to_datetime(response.json()['object']['perihelion_date'])
            print('✅ Perihelio del cometa obtenido.')
    
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response.status_code}\n{response.content}')

✅ Conectado a internet.
✅ Perihelio del cometa obtenido.


# MPC API usando astroquery.

In [66]:
# Variables
interval = 1
title = "Predefined title"
base_url = "Predefined base url"
efemerides = [['obs_date', 'delta', 'r', 'phase']]
url_ephem = "https://cgi.minorplanetcenter.net/cgi-bin/mpeph2.cgi"

# Request header
headers = {
'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Form fields needed:
data = {
    'ty': 'e',  # Return ephemerides
    'd': str((curva_de_luz_cruda_df['obs_date'].min()).date()), # Start date of ephemerides
    'l': 4001,  # Num days from start date
    'TextArea': nombre_cometa,    #Object name
    'i': interval,  # Sample interval
    'u': 'd',  # Sample interval measurements units = days
    'tit' : title,  
    'bu' : base_url,
    #... add more if needed
}

# Sending POST request with the form data
response = requests.post(url_ephem, data=data, headers=headers)
successful_download = False

# If request succeeded, process content
if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")

    print('Success in submitting the ephemeris form')
    
    data_element = soup.find('pre')
    successful_download = True

else:
    print('The ephemeris form submission was not successful')        
    
if successful_download:
    data_text = data_element.text.splitlines() # type: ignore

else:
    print("Not available ephemeris data")
    print(response.text)     # -------------------->> Only for debugging
    successful_download = False

if successful_download: 
    for line in data_text:
        parts = line.split()

        # Extract year, month, day, Delta, r and phase
        if len(parts) > 13:  # To make sure it is a data line
            year, month, day, delta, r, phase = parts[0], parts[1], parts[2], parts[8], parts[9], parts[11]
            efemerides.append(['-'.join([year, month, day]), float(delta), float(r), float(phase)]) # type: ignore
            
    print('Ephemeris data stored successfully')
    efemerides_filtrada_df = pd.DataFrame(efemerides[1::], columns= efemerides[0]) # type: ignore
    efemerides_filtrada_df['obs_date'] = pd.to_datetime(efemerides_filtrada_df.obs_date)

efemerides_filtrada_df

Success in submitting the ephemeris form
Ephemeris data stored successfully


,obs_date,delta,r,phase
0,2022-06-06,6.834,7.397,6.8
1,2022-06-07,6.823,7.390,6.8
2,2022-06-08,6.812,7.383,6.8
3,2022-06-09,6.802,7.375,6.8
4,2022-06-10,6.791,7.368,6.8
...,...,...,...,...
3996,2033-05-15,19.038,19.961,1.2
3997,2033-05-16,19.034,19.964,1.2
3998,2033-05-17,19.031,19.968,1.1
3999,2033-05-18,19.028,19.971,1.1


In [67]:
# Info del data frame ephemeris
efemerides_filtrada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4001 entries, 0 to 4000
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   obs_date  4001 non-null   datetime64[ns]
 1   delta     4001 non-null   float64       
 2   r         4001 non-null   float64       
 3   phase     4001 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 125.2 KB


# Unión de las bases de datos.

In [68]:
# Unión de las bases de datos COBS y MPC
curva_de_luz_procesada_df = curva_de_luz_cruda_df.merge(efemerides_filtrada_df, on='obs_date')
curva_de_luz_procesada_df

,obs_method_key,obs_date,magnitude,delta,r,phase
0,C,2025-05-24,17.4,4.154,5.061,5.6
1,C,2025-03-24,18.5,4.210,4.492,12.6
2,C,2025-03-23,18.1,4.216,4.482,12.7
3,C,2025-03-19,17.9,4.238,4.444,12.9
4,Z,2025-02-07,17.2,4.429,4.050,12.3
...,...,...,...,...,...,...
2221,C,2022-09-09,20.4,6.526,6.675,8.7
2222,C,2022-09-01,21.1,6.510,6.738,8.5
2223,C,2022-07-31,21.0,6.496,6.985,7.6
2224,C,2022-06-28,20.5,6.633,7.234,6.8


In [69]:
# Información del data frame curva de lus procesada
curva_de_luz_procesada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226 entries, 0 to 2225
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   obs_method_key  2226 non-null   object        
 1   obs_date        2226 non-null   datetime64[ns]
 2   magnitude       2226 non-null   float64       
 3   delta           2226 non-null   float64       
 4   r               2226 non-null   float64       
 5   phase           2226 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 104.5+ KB


In [70]:
# Reducción de la magnitud aparente y calculo del Delta t
beta = 0

curva_de_luz_procesada_df['magnitud_reducida'] = (
    curva_de_luz_cruda_df['magnitude'] 
    - 5 * np.log10(curva_de_luz_procesada_df['delta'] * curva_de_luz_procesada_df['r'])
    - (beta * curva_de_luz_procesada_df['phase'])
    )

curva_de_luz_procesada_df['delta_t'] = (curva_de_luz_procesada_df.obs_date - perihelio) # type: ignore
curva_de_luz_procesada_df['delta_t'] = curva_de_luz_procesada_df.delta_t.apply(lambda delta_t: delta_t.days)

curva_de_luz_procesada_df

,obs_method_key,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,C,2025-05-24,17.4,4.154,5.061,5.6,10.786486,397
1,C,2025-03-24,18.5,4.210,4.492,12.6,12.116391,336
2,C,2025-03-23,18.1,4.216,4.482,12.7,11.718138,335
3,C,2025-03-19,17.9,4.238,4.444,12.9,11.525325,331
4,Z,2025-02-07,17.2,4.429,4.050,12.3,10.931196,291
...,...,...,...,...,...,...,...,...
2221,C,2022-09-09,20.4,6.526,6.675,8.7,12.204508,-591
2222,C,2022-09-01,21.1,6.510,6.738,8.5,12.889440,-599
2223,C,2022-07-31,21.0,6.496,6.985,7.6,12.715938,-631
2224,C,2022-06-28,20.5,6.633,7.234,6.8,12.094557,-664


In [71]:
# Curva de luz reducida
labels = {'obs_date':'Observation Date','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='obs_date', y='magnitud_reducida', color='obs_method_key', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [72]:
# Curva de luz reducida
labels = {'delta_t':'t-Δt','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='delta_t', y='magnitud_reducida', color='obs_method_key', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [73]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 9

curva_de_luz_promediada_df = curva_de_luz_procesada_df.copy()
curva_de_luz_promediada_df['promedio_movil'] = curva_de_luz_promediada_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
curva_de_luz_promediada_df

,obs_method_key,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,C,2025-05-24,17.4,4.154,5.061,5.6,10.786486,397,NaN
1,C,2025-03-24,18.5,4.210,4.492,12.6,12.116391,336,NaN
2,C,2025-03-23,18.1,4.216,4.482,12.7,11.718138,335,NaN
3,C,2025-03-19,17.9,4.238,4.444,12.9,11.525325,331,NaN
4,Z,2025-02-07,17.2,4.429,4.050,12.3,10.931196,291,NaN
...,...,...,...,...,...,...,...,...,...
2221,C,2022-09-09,20.4,6.526,6.675,8.7,12.204508,-591,11.882479
2222,C,2022-09-01,21.1,6.510,6.738,8.5,12.889440,-599,12.042966
2223,C,2022-07-31,21.0,6.496,6.985,7.6,12.715938,-631,12.193923
2224,C,2022-06-28,20.5,6.633,7.234,6.8,12.094557,-664,12.265114


In [74]:
# Curva de luz Promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='obs_date', y='promedio_movil', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [75]:
# Curva de luz Promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='delta_t', y='promedio_movil', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v1 (Promedio corrido -> agrupación)

In [76]:
# Creación del data frame curva de luz agrupada
curva_de_luz_interna_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').max()
curva_de_luz_interna_v1_df = curva_de_luz_interna_v1_df.reset_index()

curva_de_luz_interna_v1_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686,12.364695
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664,12.265114
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631,12.193923
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599,12.042966
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591,11.882479
...,...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291,NaN
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331,NaN
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335,NaN
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336,NaN


In [77]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [78]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v1 (Promedio corrido -> agrupación)

In [79]:
# Creación del data frame curva de luz agrupada
curva_de_luz_externa_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').min()
curva_de_luz_externa_v1_df = curva_de_luz_externa_v1_df.reset_index()
curva_de_luz_externa_v1_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686,12.364695
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664,12.265114
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631,12.193923
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599,12.042966
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591,11.882479
...,...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291,NaN
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331,NaN
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335,NaN
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336,NaN


In [80]:
# Gráfica de lus promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [81]:
# Gráfica de lus promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v2 (Agrupación  -> promedio corrido)

In [82]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_max_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').max()
curva_de_luz_agrupada_max_v2_df = curva_de_luz_agrupada_max_v2_df.reset_index()
curva_de_luz_agrupada_max_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591
...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336


In [83]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_max_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark',color='obs_method_key', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [84]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 9

curva_de_luz_interna_v2_df = curva_de_luz_agrupada_max_v2_df.copy()
curva_de_luz_interna_v2_df['promedio_movil'] = curva_de_luz_interna_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
curva_de_luz_interna_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686,NaN
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664,NaN
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631,NaN
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599,NaN
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591,NaN
...,...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291,7.518393
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331,8.023725
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335,8.586857
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336,9.197082


In [85]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [86]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v2 (Agrupación  -> promedio corrido)

In [87]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v2_df = curva_de_luz_agrupada_min_v2_df.reset_index()
curva_de_luz_agrupada_min_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591
...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336


In [88]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_min_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark',color='obs_method_key', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [89]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_externa_v2_df = curva_de_luz_agrupada_min_v2_df.copy()
curva_de_luz_externa_v2_df['promedio_movil'] = curva_de_luz_externa_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
curva_de_luz_externa_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686,NaN
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664,NaN
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631,NaN
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599,NaN
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591,NaN
...,...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291,7.719750
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331,8.419887
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335,9.136915
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336,9.843293


In [90]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [91]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v1 (Mediana de cada día registrado).

In [92]:
# Creación del data frame curva de luz mediana v1
curva_de_luz_mediada_v1_df = curva_de_luz_procesada_df.groupby(by= 'obs_date').median(numeric_only= True)
curva_de_luz_mediada_v1_df = curva_de_luz_mediada_v1_df.reset_index()
curva_de_luz_mediada_v1_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2022-06-06,21.5,6.834,7.397,6.8,12.981347,-686.0
1,2022-06-28,20.5,6.633,7.234,6.8,12.094557,-664.0
2,2022-07-31,21.0,6.496,6.985,7.6,12.715938,-631.0
3,2022-09-01,21.1,6.510,6.738,8.5,12.889440,-599.0
4,2022-09-09,20.4,6.526,6.675,8.7,12.204508,-591.0
...,...,...,...,...,...,...,...
446,2025-02-07,17.2,4.429,4.050,12.3,10.931196,291.0
447,2025-03-19,17.9,4.238,4.444,12.9,11.525325,331.0
448,2025-03-23,18.1,4.216,4.482,12.7,11.718138,335.0
449,2025-03-24,18.5,4.210,4.492,12.6,12.116391,336.0


In [93]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [94]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v2 (Mediana de las dos curvas).

In [95]:
# Creación del data frame curva de luz mediana v2
curva_de_luz_mediada_v2_df = curva_de_luz_externa_v2_df.copy()
curva_de_luz_mediada_v2_df['mediana'] = (curva_de_luz_interna_v2_df['promedio_movil'] + curva_de_luz_externa_v2_df['promedio_movil'])/2
curva_de_luz_mediada_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil,mediana
0,2022-06-06,C,21.5,6.834,7.397,6.8,12.981347,-686,NaN,NaN
1,2022-06-28,C,20.5,6.633,7.234,6.8,12.094557,-664,NaN,NaN
2,2022-07-31,C,21.0,6.496,6.985,7.6,12.715938,-631,NaN,NaN
3,2022-09-01,C,21.1,6.510,6.738,8.5,12.889440,-599,NaN,NaN
4,2022-09-09,C,20.4,6.526,6.675,8.7,12.204508,-591,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
446,2025-02-07,Z,17.2,4.429,4.050,12.3,10.931196,291,7.719750,7.619072
447,2025-03-19,C,17.9,4.238,4.444,12.9,11.525325,331,8.419887,8.221806
448,2025-03-23,C,18.1,4.216,4.482,12.7,11.718138,335,9.136915,8.861886
449,2025-03-24,C,18.5,4.210,4.492,12.6,12.116391,336,9.843293,9.520188


In [96]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='obs_date', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [97]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='delta_t', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Comparación de las curvas de luz v1 (Promedio corrido -> agrupación)

In [98]:
# Gráfica de luz promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.obs_date, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente_v2', marker=dict(color='green', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo_v2', marker=dict(color='blue', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana_v2', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

In [99]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.delta_t, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.delta_t, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.delta_t, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

# Comparación de las curvas de luz v2 (Agrupación -> promedio corrido)

In [100]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

In [101]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_procesada_df.delta_t, y=curva_de_luz_procesada_df.magnitud_reducida, mode='markers', name='Envolvente', marker=dict(line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.delta_t, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.delta_t, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.delta_t, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()